# Chapter 4: Building Ground Truth — Ingestion and the GMS Expert Store

**Learning objectives:**
- Ingest a markdown document into a GMS expert store on GPU
- Inspect entities, relations, triples and ENM entries
- Understand the ingestion pipeline: parse → extract → train → populate ENM
- Explore store persistence and the files written to disk
- Extract and analyze entity embeddings on the unit hypersphere

---
## Preamble

This notebook uses two layers of the GMSH stack:
- `knowlytix.benchmark.ingest` — the markdown parser that extracts triples, ENM entries and phase encoders from a source document.
- `knowlytix.harness.testing` — the DOE benchmark harness that orchestrates ingestion, training and store persistence via `DOEGMSBenchmark`.

Chapters 1–3 treated the GMS store as a black box: we ingested a document and immediately started querying it. This chapter opens the box and walks through every stage of the pipeline.

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

---

## 4.1 The Ingestion Pipeline

The path from a source document to a queryable GMS expert store has four stages:

| Stage | Input | Output | Key Function |
|-------|-------|--------|-------------|
| **Parse** | Markdown text | `DocumentGraph` | `ingest_markdown()` |
| **Adapt** | `DocumentGraph` | Index tensors | `GraphToGMS()` |
| **Train** | Index tensors | `GeometricKnowledgeGraph` | `train_gms()` |
| **Populate** | ENM entries | `ExactNumericalMemory` | `populate_enm()` |

The `DOEGMSBenchmark.ingest()` method runs all four stages behind the scenes. We start by examining each stage individually, then show the integrated pipeline.

### Stage 1: Parsing the Source Document

The parser reads a markdown file and extracts structured knowledge. It recognizes three kinds of content:

- **Tables** → triples (`has_column` relations) plus ENM entries for numeric cells
- **Bullet lists** → key-value triples (`has_detail` relations)
- **Section headings** → hierarchical `in_section` triples

Let us parse the model risk assessment document directly.

In [ ]:
from importlib.resources import files
from knowlytix.benchmark.ingest import ingest_markdown

MD_PATH = str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))

# Pin the ingest mode explicitly so the chapter's published numbers
# (807 triples / 261 ENM) reproduce byte-for-byte. The library
# default is 'regex'; the orchestrator (DocGMSConfig) defaults to
# 'hybrid' but we use the standalone parser here.
doc_graph = ingest_markdown(MD_PATH, mode="regex")

print(f"Triples:         {len(list(doc_graph.triples))}")
print(f"ENM entries:     {len(doc_graph.enm)}")
print(f"Phase encoders:  {len(doc_graph.phase_encoders)}")
print(f"Document title:  {doc_graph.metadata.get('title', '(none)')}")
print(f"Ingest mode:     {doc_graph.metadata.get('ingest_mode')}")


From 14,544 characters of markdown the parser extracted 807 triples and 261 exact numeric values. The `DocumentGraph` is a lightweight in-memory structure — not yet geometric. It holds the raw knowledge graph that will be embedded on the hypersphere in Stage 3.

The `stats()` method breaks down the extraction by section and relation type.

In [ ]:
stats = doc_graph.stats()

# ENM entries by section
print("ENM entries by section:")
print(f"  {'Section':40s} {'Count':>6s}")
print("  " + "-" * 50)
for section, count in stats["enm_types"].items():
    print(f"  {section:40s} {count:>6d}")

print(f"\nTotal ENM entries: {stats['enm_entries']}")
print(f"Total triples:     {stats['triples']}")
print(f"Relation types:    {len(stats['relations'])}")
print(f"Phase encoders:    {stats['phase_encoders']}")

### What Gets Extracted

The parser classifies each table column as numeric, boolean or label. Numeric values are stored twice: once as a triple (`MDL-001, has_test_auc, 0.8412`) and again in the ENM register with a SHA-256 integrity hash. This dual storage means the GMS can answer approximate geometric queries *and* return exact IEEE 754 values when precision matters.

The first 20 triples show the executive summary bullets followed by the model inventory table:

In [ ]:
triples = list(doc_graph.triples)

print("First 20 triples:")
for h, r, t in triples[:20]:
    print(f"  ({h}, {r}, {t})")

The executive summary bullets produce `has_detail` triples ("Organization has_detail Meridian Financial Technologies, Inc."). Table rows produce typed relations derived from column headers (`has_tier`, `has_model_name`, `has_domain`). Every entity also receives an `in_section` triple linking it to the document section where it was found.

Let us look at how triples are distributed across the 8 document sections.

In [ ]:
# Section-level triple counts
section_triples = [(h, r, t) for h, r, t in doc_graph.triples if r == "in_section"]
section_counts = Counter(t for h, r, t in section_triples)

print(f"{'Section':35s} {'Entities':>8s}")
print("-" * 45)
for sec, cnt in section_counts.most_common():
    print(f"{sec:35s} {cnt:>8d}")

In [ ]:
# Relation distribution (top 20)
rel_counts = Counter(r for h, r, t in doc_graph.triples)

print(f"Relation distribution (top 20 of {len(rel_counts)}):")
print(f"{'Relation':35s} {'Count':>6s}")
print("-" * 43)
for rel, cnt in rel_counts.most_common(20):
    print(f"{rel:35s} {cnt:>6d}")

The `in_section` relation is the most frequent (92 triples) because every entity is tagged with its section. The 12 model-level relations (`has_tier`, `has_model_name`, etc.) each appear exactly 12 times — one per model. Special relations like `fails` and `is_weak` capture boolean pass/fail columns from the fairness assessment table. Note that triples are now deduplicated at ingestion time — an entity appearing in multiple tables within the same section produces only one `in_section` triple per distinct section.

### Phase Encoders

The parser also infers **phase encoders** from column names that match common metric patterns (AUC, accuracy, ratio, rate, etc.). A phase encoder maps a numeric value to a phase angle in $[0, \pi]$, enabling the GMS to reason about *relative magnitude* — "is this AUC above threshold?" — without relying on string comparison.

In [ ]:
# Phase encoders detected from column names
print("Phase encoders:")
for name, pe in doc_graph.phase_encoders.items():
    print(f"  {name:15s}  range=[{pe.v_min:.2f}, {pe.v_max:.2f}]")

# Demonstrate phase encoding on AUC values
import math
auc_pe = doc_graph.phase_encoders["AUC"]

print()
print("AUC phase encoding:")
print(f"  {'Value':>6s}  {'Phase (rad)':>12s}  {'Phase (deg)':>12s}")
print("  " + "-" * 34)
for val in [0.50, 0.70, 0.80, 0.84, 0.90, 0.95, 1.00]:
    phase = auc_pe.encode(val)
    print(f"  {val:>6.2f}  {phase:>12.4f}  {phase * 180 / math.pi:>11.1f}°")

An AUC of 0.50 (random chance) maps to 90°, while a perfect 1.00 maps to 180°. The phase encoding preserves ordering: if AUC$_A$ > AUC$_B$, then $\phi_A > \phi_B$. This enables threshold checks without string parsing.

The `check_threshold()` method uses the phase encoding to verify inequalities:

In [ ]:
# Threshold checks via phase encoding
print("Threshold checks:")
checks = [
    ("AUC", 0.8412, 0.80, "ge", "MDL-001 Test AUC >= 0.80"),
    ("AUC", 0.7523, 0.80, "ge", "MDL-007 Test AUC >= 0.80"),
    ("AUC", 0.9100, 0.90, "gt", "MDL-002 approx > 0.90"),
]
for enc_name, value, limit, op, label in checks:
    sat, margin = doc_graph.check_threshold(enc_name, value, limit, op)
    status = "PASS" if sat else "FAIL"
    print(f"  {label:35s}  {status}  (margin={margin:+.4f})")

---

## Stage 2: Adapting the DocumentGraph to GMS Format

The `GraphToGMS` adapter converts string-based triples into integer index tensors that PyTorch can process. It builds bidirectional vocabularies (entity $\leftrightarrow$ index, relation $\leftrightarrow$ index) and pre-computes per-relation tail sets for type-constrained negative sampling during training.

In [ ]:
from knowlytix.core.train_finstructbench import GraphToGMS

adapter = GraphToGMS(doc_graph)

The adapter has converted 807 string triples into three aligned integer tensors. Let us inspect the tensor shapes and a few sample mappings.

In [ ]:
print("Training tensors:")
print(f"  heads shape:     {adapter.heads.shape}  dtype={adapter.heads.dtype}")
print(f"  relations shape: {adapter.relations.shape}")
print(f"  tails shape:     {adapter.tails.shape}")

print("\nSample index triples (first 5):")
for i in range(5):
    h, r, t = adapter.heads[i].item(), adapter.relations[i].item(), adapter.tails[i].item()
    h_name = adapter.idx_to_entity[h]
    r_name = adapter.idx_to_relation[r]
    t_name = adapter.idx_to_entity[t]
    print(f"  [{h:>3d}, {r:>2d}, {t:>3d}]  ({h_name}, {r_name}, {t_name})")

print("\nType-constrained tail pools (5 relations):")
for r_idx in list(adapter.tail_per_relation.keys())[:5]:
    r_name = adapter.idx_to_relation[r_idx]
    pool = adapter.tail_per_relation[r_idx]
    print(f"  {r_name:25s}  {len(pool):>3d} valid tails")

---

## Stage 3: Training the Geometric Knowledge Graph

The `train_gms` function trains a dual-space geometric model on the indexed triples using manifold-aware optimization. The two embedding spaces serve different purposes:

- **v-space** (semantic): embeddings on $S^{m-1}$ with learned relation operators for triple scoring and link prediction
- **u-space** (logical): embeddings for tension energy and contradiction detection

Training is controlled by two configuration objects:
- `GeometryConfig`: embedding dimensions ($d_v$, $d_u$, $m$, $d$)
- `LossConfig`: loss weights, margin, temperature (see Appendix F for full details)

In [ ]:
import math
from knowlytix.core.config import GeometryConfig, LossConfig, CapLossConfig
from knowlytix.core.train_finstructbench import train_gms

# --- Uncomment to force retraining (deletes existing store) ---
# shutil.rmtree("gmsh_testing_ext/lab_stores/ch04_ground_truth",
#               ignore_errors=True)

# Geometry: dual 256-d embeddings projected to 128-d shared sphere
geometry = GeometryConfig(d_v=256, d_u=256, m=128, d=128)

# Loss: validated defaults (see Appendix F, Table F.2)
loss_cfg = LossConfig()

# Cap admissibility (HARDENING_PLAN.md §2.1; ch04.tex §Cap admissibility).
# cap_use_diag=True matches the agent-builder ds_policy fixture.
cap_cfg = CapLossConfig(use_diag=True, rho_max=math.pi / 2)

print("Loss configuration:")
print(f"  lambda_geo={loss_cfg.lambda_geo}, lambda_tension={loss_cfg.lambda_tension}")
print(f"  lambda_neg={loss_cfg.lambda_neg}, gamma={loss_cfg.gamma}")
print(f"  cap rho_max={cap_cfg.rho_max:.4f}, use_diag={cap_cfg.use_diag}")

# Train the geometric knowledge graph (cap-on)
model = train_gms(
    adapter, DEVICE,
    epochs=500,
    batch_size=256,
    num_neg=32,
    geometry=geometry,
    loss=loss_cfg,
    loss_mode="cap",
    cap=cap_cfg,
)
model.eval()
print("\nTraining complete. Model in eval mode.")

The training log shows the combined loss decreasing over 200 epochs. Three loss components contribute:

- **L_pos**: geodesic distance for positive triples --- drives related entities together via learned relation operators
- **L_neg**: margin penalty for corrupted triples --- pushes unrelated entities apart
- **L_te**: tension energy supervision in u-space --- teaches the model to distinguish agreement from contradiction

Let us verify the trained model's geometric structure.

In [ ]:
# Verify model architecture
print("Model architecture:")
print(f"  Entities:   {model.num_entities}")
print(f"  Relations:  {model.num_relations}")
print(f"  Rotors:     {len(model.relation_rotors)} relation operators")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Verify embeddings lie on the unit sphere
with torch.no_grad():
    all_idx = torch.arange(adapter.num_entities, device=DEVICE)
    v_emb = model.dual_emb.project_v(all_idx)
    u_emb = model.dual_emb.project_u(all_idx)
    v_norms = v_emb.norm(dim=1)
    u_norms = u_emb.norm(dim=1)

print("\nManifold verification:")
print(f"  v-space: shape={v_emb.shape}, max |norm-1| = {(v_norms - 1).abs().max():.2e}")
print(f"  u-space: shape={u_emb.shape}, max |norm-1| = {(u_norms - 1).abs().max():.2e}")

# Verify a relation operator is orthogonal (det = 1)
with torch.no_grad():
    rotor = model.relation_rotors[0]
    R = rotor.get_rotor()  # rotation matrix
    orth_err = (R @ R.T - torch.eye(R.shape[0], device=DEVICE)).abs().max()
    det = torch.linalg.det(R)
print("\nrelation operator check (relation 0):")
print(f"  Orthogonality error: {orth_err:.2e}")
print(f"  Determinant:         {det:.6f} (should be 1.0)")


---

## Stage 4: Populating the ENM Register

The final stage fills the **Exact Numerical Memory** register with IEEE 754 values and SHA-256 integrity hashes. This gives the store lossless numeric lookup alongside the lossy (but rich) geometric embeddings.

In [ ]:
from knowlytix.core.train_finstructbench import populate_enm

enm = populate_enm(doc_graph, adapter)

# Verify integrity
keys = enm.keys()
n_ok = sum(1 for k in keys if enm.verify_integrity(k))
print(f"Integrity check: {n_ok}/{len(keys)} entries passed")

---

## Assembling and Saving the GMSExpertStore

With all four stages complete we now have the individual components: a trained `GeometricKnowledgeGraph`, a `GraphToGMS` adapter, an `ExactNumericalMemory` register and the original `DocumentGraph`. The `GMSExpertStore` class unifies these into a single persistent object that also includes the transport layer (multi-hop reasoning), compression memory (approximate storage) and memory router (exact vs. approximate dispatch).

In [ ]:
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.core.graph.transport import RelationalTransport
from knowlytix.core.memory.compression import CompressionMemory
from knowlytix.core.memory.router import MemoryRouter

config_doc = DocGMSConfig(store_path="gmsh_testing_ext/lab_stores/ch04_ground_truth")

store = GMSExpertStore(config_doc, device=DEVICE)
store.model = model
store.adapter = adapter
store.enm = enm
store.doc_graph = doc_graph
store.transport = RelationalTransport(model)
store.compression = CompressionMemory(k=config_doc.memory.k, d=geometry.m)
store.router = MemoryRouter(enm, store.compression)
with open(MD_PATH) as f:
    store.markdown = f.read()

# Persist to disk
store.save()

print()
stats = store.stats()
print("Store contents:")
for k, v in stats.items():
    print(f"  {k:15s} {v}")

---

## Post-Training Diagnostics

Before using the store for testing, we run the nine-check diagnostic suite (see Appendix F for thresholds). Three structural invariant checks (S1--S3) verify that the relation operators are orthogonal, the projection matrices are valid and all embeddings lie on the unit sphere. Six primitive checks (P1--P6) verify that each GMS operation works correctly.

In [ ]:
from knowlytix.core.diagnostics import GMSDiagnostics

diag = GMSDiagnostics(model, adapter, enm, doc_graph, DEVICE)
report = diag.run_all()
GMSDiagnostics.print_report(report)

All nine checks should pass, confirming that the store is ready for use. If any primitive check fails, consult Appendix F (Table F.5) for troubleshooting guidance.

---

With the store built from scratch, verified and saved, we now show the **convenience wrapper** that automates these same steps. `DOEGMSBenchmark.ingest()` runs Stages 1--4, assembles the store and caches it to disk --- all in a single call. Since our store already exists on disk, the wrapper will simply load it.

## Lab 4.1 -- The Convenience Wrapper: DOEGMSBenchmark

In practice you will rarely call the four stages manually. The `DOEGMSBenchmark` class wraps the entire pipeline into a single `ingest()` call and handles caching --- if a store exists on disk it loads instantly, otherwise it trains from scratch. Since we already saved our store above, this call simply loads it.

In [ ]:
from importlib.resources import files
import math
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    ingest_mode="regex",
    cap_enabled=True,
    cap_use_diag=True,
    cap_rho_max=math.pi / 2,
    device=str(DEVICE),
)
assert config.enable_tracing is False

bench = DOEGMSBenchmark(config)
store = bench.ingest()  # loads from disk (no retraining)

stats = store.stats()
print("Store contents:")
for k, v in stats.items():
    print(f"  {k:15s} {v}")

The wrapper detected the existing store and loaded it without retraining. This caching behavior is what makes notebook iteration fast: the training cost is paid once, and all subsequent calls are instant loads.

### Browsing Triples with Geodesic Scores

Each triple is now embedded geometrically. The `score_triple()` method returns the geodesic distance between the operator-transported head and the tail. Lower scores indicate stronger geometric agreement.

In [ ]:
# Browse MDL-001 triples with geodesic scores
triples_001 = store.query_triples(head="MDL-001")

print(f"MDL-001 has {len(triples_001)} triples. First 15:")
print(f"  {'Head':>8s}  {'Relation':>20s}  {'Tail':>25s}  {'Geodesic':>9s}")
print("  " + "-" * 68)
for h, r, t in triples_001[:15]:
    score = store.score_triple(h, r, t)
    if score is not None:
        print(f"  {h:>8s}  {r:>20s}  {t:>25s}  {score:>9.4f}")
    else:
        print(f"  {h:>8s}  {r:>20s}  {t:>25s}  {'N/A':>9s}")

The geodesic scores reveal meaningful variation across relation types. Structural triples like `in_section` and `has_retrain_freq` tend to score well because they have consistent patterns across entities. Metric triples like `has_test_auc` and `has_train_auc` are harder to embed precisely because numeric string tails are less structurally distinctive.

The key insight: **geodesic distance is not binary.** A score of ~1.2 means the GMS has high confidence in the triple. A score of ~1.6 (close to $\pi/2$) means the model considers the triple plausible but is less certain. This continuous signal is what makes GMS verification richer than string matching.

### The Adapter: Vocabularies and Index Mappings

Between parsing and training sits the **adapter** (`GraphToGMS`). It converts string-based triples into integer index tensors that PyTorch can process. The adapter maintains bidirectional mappings between entity names and indices.

In [ ]:
adapter = store.adapter

print(f"Entities:  {adapter.num_entities}")
print(f"Relations: {adapter.num_relations}")

# Sample entity names (sorted alphabetically)
entities = list(adapter.entity_to_idx.keys())
print(f"\nSample entities (first 10 of {len(entities)}):")
for e in entities[:10]:
    print(f"  [{adapter.entity_to_idx[e]:>3d}] {e}")

# Model entity indices
print("\nModel entity indices:")
for m in ["MDL-001", "MDL-002", "MDL-003", "MDL-012"]:
    print(f"  {m} -> index {adapter.entity_to_idx[m]}")

In [ ]:
# All 66 relation types
relations = list(adapter.relation_to_idx.keys())

print(f"All {len(relations)} relations:")
for r in relations:
    print(f"  [{adapter.relation_to_idx[r]:>2d}] {r}")

The 508 entities include model IDs, metric values, dollar amounts, dates, status labels and section names — everything that appears as a head or tail in any triple. The 66 relations correspond to table column headers (`has_test_auc`, `has_risk_rating`), structural links (`in_section`) and boolean assessments (`passes`, `fails`, `is_weak`, `is_strong`).

---

## 4.2 Exact Numerical Memory

Geometric embeddings are powerful for approximate reasoning but they are lossy — you cannot recover the exact value 0.8412 from a 128-dimensional unit vector. The **ENM register** solves this by storing every numeric value in IEEE 754 double precision with a SHA-256 integrity hash. When the testing pipeline needs to verify an exact number, it queries the ENM directly rather than relying on the geometric embedding.

In [ ]:
enm = store.enm
print(f"ENM entries: {enm.size()}")

# ENM entries are keyed by (section, entity_id)
# Show the 5 section categories
cat_counts = Counter(key.type for key in enm.keys())

print("\nENM entries by section:")
print(f"  {'Section':40s} {'Count':>6s}")
print("  " + "-" * 50)
for cat, cnt in cat_counts.most_common():
    print(f"  {cat:40s} {cnt:>6d}")

In [ ]:
# Performance metrics for MDL-001 through MDL-004
perf_keys = [k for k in enm.keys() if k.type == "performance_metrics"]

print("Performance metrics (first 24 — MDL-001 through MDL-004):")
print(f"  {'Entity/Metric':55s} {'Value':>10s}")
print("  " + "-" * 67)
for key in perf_keys[:24]:
    val = enm.read_exact(key)
    print(f"  {key.id:55s} {float(val):>10.4f}")

Each ENM entry stores the exact IEEE 754 representation of the numeric value. The key is an `ENMKey(type, id, timestamp)` triple where `type` is the document section and `id` is a composite string that uniquely identifies the value within its section.

### SHA-256 Integrity Verification

Every ENM entry carries a SHA-256 hash computed from the raw bytes of the stored float64 value. This means we can detect any corruption — accidental or adversarial — after the fact.

In [ ]:
# Verify integrity for all 261 entries
keys = enm.keys()
n_ok = sum(1 for k in keys if enm.verify_integrity(k))

print(f"Integrity verification: {n_ok}/{len(keys)} entries passed")

# Demonstrate a single verification -- use an actual stored key
# (ENMKey includes a timestamp, so we retrieve the key from the store)
perf_keys = [k for k in keys if k.type == "performance_metrics"
             and "Test AUC" in k.id]
test_auc_key = perf_keys[0]

val = enm.read_exact(test_auc_key)
ok = enm.verify_integrity(test_auc_key)
print(f"\nExample: ({test_auc_key.type!r}, {test_auc_key.id!r})")
print(f"  Value:     {float(val):.4f}")
print(f"  Integrity: {ok}")

All 261 entries pass. In a production deployment the ENM integrity check runs automatically during claim verification — if an agent claims "Test AUC is 0.8412" and the hash does not match, the claim is rejected regardless of geometric similarity.

---

## 4.3 Persistence: The Store on Disk

After ingestion the store is saved to disk so that subsequent runs can load it without retraining. Let us inspect the files.

In [ ]:
store_path = "gmsh_testing_ext/lab_stores/ch04_ground_truth"

print(f"Store directory: {store_path}/")
print(f"{'File':25s}  {'Size':>10s}")
print("-" * 38)
for f in sorted(os.listdir(store_path)):
    full = os.path.join(store_path, f)
    if os.path.isfile(full):
        size = os.path.getsize(full)
        if size > 1024 * 1024:
            print(f"{f:25s}  {size / 1024 / 1024:.1f} MB")
        elif size > 1024:
            print(f"{f:25s}  {size / 1024:.1f} KB")
        else:
            print(f"{f:25s}  {size} B")
    else:
        print(f"{f:25s}  (directory)")

# Documents subdirectory
doc_dir = os.path.join(store_path, "documents")
if os.path.exists(doc_dir):
    print("\n  documents/:")
    for f in os.listdir(doc_dir):
        size = os.path.getsize(os.path.join(doc_dir, f))
        print(f"    {f:25s}  {size / 1024:.1f} KB")


The store consists of eight files:

| File | Contents |
|------|----------|
| `model.pt` | PyTorch state dict — entity embeddings, relation operators, projection matrices |
| `model_dims.json` | Number of entities and relations (needed to reconstruct the model architecture) |
| `adapter.json` | Entity-to-index and relation-to-index vocabularies |
| `enm.json` | Exact numeric values with section/ID keys |
| `triples.json` | All knowledge triples from the document graph |
| `phase.json` | Phase encoder ranges (v_min, v_max per metric type) |
| `metadata.json` | Ingestion history, timestamps, document list |
| `documents/combined.md` | Original source document (for provenance) |

The `model.pt` file (5.4 MB) dominates — it contains the learned embeddings and operators for 508 entities and 66 relations at $m = 128$ projected dimension. Everything else is lightweight JSON.

In [ ]:
from importlib.resources import files
# Verify that loading from disk reproduces the same store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config2 = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)
assert config2.enable_tracing is False

bench2 = DOEGMSBenchmark(config2)
store2 = bench2.ingest()  # loads from disk (no retraining)

stats1 = store.stats()
stats2 = store2.stats()
print("Loaded store matches fresh build:")
for key in ("entities", "relations", "triples", "enm_entries"):
    flag = "OK" if stats1[key] == stats2[key] else "MISMATCH"
    print(f"  {key:13s} fresh={stats1[key]}  loaded={stats2[key]}  {flag}")

# Verify a geodesic score is identical
s1 = store.score_triple("MDL-001", "has_risk_rating", "High")
s2 = store2.score_triple("MDL-001", "has_risk_rating", "High")
print("\nscore_triple('MDL-001','has_risk_rating','High'):")
print(f"  Original: {s1:.6f}")
print(f"  Loaded:   {s2:.6f}")
print(f"  Match:    {abs(s1 - s2) < 1e-6}")


The loaded store is byte-identical to the original. The ingestion log shows "Loading existing GMS store" instead of "Ingesting" — no retraining needed. This makes subsequent experiments fast: the training cost is paid once.

---

## Lab 4.2 -- Explore Entity Embeddings

The GMS places every entity on the unit hypersphere $S^{m-1}$ in $\mathbb{R}^m$ where $m = 128$ (the projected dimension). Since all embeddings have unit norm, cosine similarity equals the dot product. Let us extract the embeddings and see what the geometry tells us about the 12 models.

In [ ]:
# Extract v-space (semantic) embeddings for all entities
model_gkg = store.model
adapter = store.adapter

with torch.no_grad():
    all_idx = torch.arange(adapter.num_entities, device=DEVICE)
    v_emb = model_gkg.dual_emb.project_v(all_idx)  # (508, 64)

print(f"v-space embeddings: shape={v_emb.shape}")
print(f"All on unit sphere:  max |norm - 1| = {(v_emb.norm(dim=1) - 1.0).abs().max().item():.2e}")

In [ ]:
# Cosine similarity matrix for the 12 models
model_ids = [f"MDL-{i:03d}" for i in range(1, 13)]
model_idx = [adapter.entity_to_idx[m] for m in model_ids]
model_emb = v_emb[model_idx]  # (12, 64)

cos_sim = model_emb @ model_emb.T  # unit norm -> dot product = cosine sim

print("Cosine similarity matrix (12 models):")
print(f"{'':>8s}", end="")
for m in model_ids:
    print(f"{m[-3:]:>7s}", end="")
print()
for i, mi in enumerate(model_ids):
    print(f"{mi:>8s}", end="")
    for j in range(12):
        print(f"{cos_sim[i, j].item():>7.3f}", end="")
    print()

The similarity matrix reveals meaningful structure. With $m = 128$ projected dimensions, the cosine similarities are lower in absolute magnitude than they would be in a smaller space — this is expected because high-dimensional unit vectors tend to be nearly orthogonal. The patterns are still informative:

- **MDL-001 and MDL-008** (Credit Decisioning and LGD-Ensemble) have relatively high similarity (0.42) — both are Tier 1/2 models with interconnected dependencies (credit scores segment LGD pools).
- **MDL-001 and MDL-009** (Credit and Probability of Default) cluster at 0.41 — MDL-009 feeds directly into MDL-001 per the dependency table.
- **MDL-003 and MDL-012** (AML and Document Classification) are similar (0.47) — MDL-012's doc classification feeds AML alerts, and both are Degraded.
- **MDL-005 and MDL-011** (Customer Analytics and Segmentation) cluster at 0.39 — both are in the customer domain and MDL-011 defines churn cohorts for MDL-005.

The GMS has learned that models sharing dependency chains, risk tier, status and relation patterns should be geometrically close — not because we told it to, but because the manifold-aware training naturally clusters entities with similar neighborhoods in the knowledge graph.

In [ ]:
# Tension energy between selected entity pairs
print("Tension energy (selected pairs):")
print(f"  {'Entity A':>12s}  {'Entity B':>12s}  {'E(a,b)':>8s}  {'Interpretation':>20s}")
print("  " + "-" * 58)

pairs = [
    ("MDL-001", "MDL-009", "related models"),
    ("MDL-001", "MDL-010", "distant models"),
    ("High", "Low", "opposing labels"),
    ("Stable", "Degraded", "opposing status"),
]
for a, b, label in pairs:
    te = store.tension_energy(a, b)
    if te is not None:
        print(f"  {a:>12s}  {b:>12s}  {te:>8.4f}  {label:>20s}")

## Exercises

### Exercise 4.1

Use `store.query_triples(head="MDL-005")` to list all triples for model MDL-005. How many triples does it have? Compare the geodesic scores for its `has_risk_rating` and `has_status` triples against MDL-001's. Are the distances consistent with what you know about the two models?

### Exercise 4.2

The ENM stores 261 entries across 5 sections. Write a loop that reads all `performance_metrics` entries and groups them by metric type (Train AUC, Test AUC, Gini, etc.). Which metric has the widest range across the 12 models? Which has the narrowest?

### Exercise 4.3

Extract the v-space embeddings for all entities whose name starts with `"MDL-"`. Use `sklearn.decomposition.PCA` (or `torch.pca_lowrank`) to project them to 2D and plot the result with `matplotlib`. Do the clusters align with risk tier, domain or status?

In [ ]:
# === Exercise 4.1: Your code here ===
# Starter: uncomment and run
# triples_005 = store.query_triples(head="MDL-005")
# print(f"MDL-005 has {len(triples_005)} triples")
# for h, r, t in triples_005[:10]:
#     score = store.score_triple(h, r, t)
#     print(f"  ({h}, {r}, {t})  geodesic={score:.4f}")

In [ ]:
# === Exercise 4.2: Your code here ===


In [ ]:
# === Exercise 4.3: Your code here ===


---

## Three-Mode Ingestion Demonstration

The parser supports three ingestion modes — `regex`, `hybrid`, and `llm_only` — corresponding to the three-way comparison reported in the FinStructBench paper §6.5. The cells below run the same source document through each mode and verify the contract:

* **`regex`** is deterministic and reproduces the chapter's numbers byte-for-byte.
* **`hybrid`** is a strict superset of regex on numeric values: every shared ENM key carries the same value (100% bit-identical), and every regex triple is preserved. Adds prose-extracted triples regex cannot reach.
* **`llm_only`** runs the LLM end to end with no regex backbone. Recovers ≈12.5% of regex ENM entries with near-zero overlap (paper §6.5) — useful for paper reproduction, not for production.

We use a deterministic mock callable below so the cells run without an API key. Replace `mock_llm` with `make_anthropic_callable()` to run against real Claude.


In [ ]:
# Deterministic mock LLM callable (CI-hermetic). Swap with
# `from knowlytix.benchmark.llm_caller import make_anthropic_callable`
# and `mock_llm = make_anthropic_callable()` to run against Claude.

def mock_llm(prompt: str) -> str:
    if "Classify each column" in prompt:
        return '{"Model": "entity", "AUC": "numeric", "Status": "boolean"}'
    if "Which column contains the primary entity identifier" in prompt:
        return "Model"
    if "Extract all factual relationships" in prompt:
        return '[]'
    if "Extract ALL data from this table" in prompt:
        return '{"enm": [], "triples": []}'
    if "Extract all key-value pairs" in prompt:
        return '{"enm": [], "triples": []}'
    return "{}"


In [ ]:
# Build the document graph in each of the three modes.
g_regex   = ingest_markdown(MD_PATH, mode="regex")
g_hybrid  = ingest_markdown(MD_PATH, mode="hybrid",   llm_callable=mock_llm)
g_llmonly = ingest_markdown(MD_PATH, mode="llm_only", llm_callable=mock_llm)

print(f"{'mode':<10s}  {'ENM':>5s}  {'triples':>7s}  {'phase':>5s}")
print('-' * 36)
for name, g in [("regex", g_regex), ("hybrid", g_hybrid), ("llm_only", g_llmonly)]:
    s = g.stats()
    print(f"{name:<10s}  {s['enm_entries']:>5d}  {s['triples']:>7d}  {len(s['phase_encoders']):>5d}")


In [ ]:
# Verify the byte-equality and superset invariants the test suite asserts.

# 1. hybrid is byte-identical to regex on every shared ENM key.
shared = set(g_regex.enm.keys()) & set(g_hybrid.enm.keys())
drift = [k for k in shared if g_regex.enm[k].value != g_hybrid.enm[k].value]
print(f"hybrid value drift on shared ENM keys: {len(drift)} / {len(shared)}  (expect 0)")

# 2. hybrid never drops a regex triple.
missing = set(g_regex.triples) - set(g_hybrid.triples)
print(f"regex triples missing from hybrid:      {len(missing)}  (expect 0)")

# 3. mode=hybrid without callable raises ValueError.
try:
    ingest_markdown(MD_PATH, mode="hybrid")
    print("ERROR: expected ValueError")
except ValueError as e:
    print(f"hybrid w/o callable raises ValueError: {str(e)[:60]}…")


---

## Summary

In this chapter we built a GMS expert store from scratch, walking through every stage of the pipeline:

1. **Parsing** (Stage 1) extracts structured knowledge from markdown --- tables become triples and ENM entries, bullets become key-value facts and section headings create hierarchical links. From 14,544 characters the parser produced 807 deduplicated triples, 261 ENM entries and 7 phase encoders.
2. **Adapting** (Stage 2) converts string-based triples into integer index tensors via `GraphToGMS`, building bidirectional vocabularies for 508 entities and 66 relations with type-constrained tail pools for negative sampling.
3. **Training** (Stage 3) uses manifold-aware optimization to embed the knowledge graph on $S^{m-1}$ with $m = 128$. The dual-space architecture trains v-space (semantic) and u-space (logical) with separate loss components.
4. **ENM population** (Stage 4) stores every numeric value in IEEE 754 with SHA-256 integrity --- all 261 entries pass verification.
5. **Assembly** packages the trained model, adapter, ENM, transport, compression and router into a `GMSExpertStore` and persists it to disk (8 files, ~5.4 MB).
6. **Diagnostics** (9 checks) confirm that all structural invariants hold and all six GMS primitives work correctly.
7. **Entity embeddings** on the unit hypersphere capture meaningful structure: models sharing dependency chains and status cluster together, and tension energy separates opposing labels.

The `DOEGMSBenchmark.ingest()` wrapper automates Steps 1--5 and handles caching, but understanding the individual stages is essential for troubleshooting, custom configurations and advanced use cases.

**Next:** Chapter 5 shows how to generate test questions from this geometric structure --- each question backed by a provable ground truth from the GMS primitives.